In [8]:
import torch
import torch.nn as nn
from typing import Optional

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.base_layer = nn.Linear(10, 10)
        self.feature_layer = nn.Linear(5, 10)
        self.output_layer = nn.Linear(10, 1)
        
    def forward(self, x: torch.Tensor, feature: Optional[torch.Tensor] = None):
        # 基本的な処理
        x = self.base_layer(x)
        
        # featureがNoneかどうかで処理を分岐
        if feature is not None:
            # featureがある場合の処理
            feature_out = self.feature_layer(feature)
            x = x + feature_out
        else:
            # featureがない場合の処理（必要に応じて）
            pass
        
        # 最終出力
        return self.output_layer(x)

    @torch.jit.export
    def forward_inference(self, x: torch.Tensor, feature: Optional[torch.Tensor] = None):
        return self.forward(x, feature)

In [9]:
# モデルの初期化
model = MyModel()

In [10]:
# スクリプト化
scripted_model = torch.jit.script(model)

# エクスポート
scripted_model.save("my_model.pt")

# テストしてみる（Pythonで）
x = torch.ones(1, 10)
# x = torch.randn(1, 10)
print(x)

feature = torch.ones(1, 5)  # または None
# feature = torch.randn(1, 5)  # または None
print(feature)

# 両方のケースをテスト
output1 = scripted_model.forward_inference(x, feature)
output2 = scripted_model.forward_inference(x, None)

tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])
tensor([[1., 1., 1., 1., 1.]])


In [11]:
print(f"{output1=}")
print(f"{output2=}")

output1=tensor([[-0.5617]], grad_fn=<AddmmBackward0>)
output2=tensor([[-0.3922]], grad_fn=<AddmmBackward0>)
